# Automotive C Code Analyzer with Custom Coding Standards Compliance and Unit Test Generation

This notebook demonstrates a multi-agent system for automotive C code analysis using Strands agents deployed on Amazon Bedrock AgentCore Runtime with inbound authentication.

## Overview

The system implements a graph multi-agent pattern with 2 steps:
1. **Automotive Coding Standards Analysis**: Analyzes C code for custom automotive coding standards violations
2. **Unit Test Generation**: If no severe violations are found, generates comprehensive unit tests

### Architecture

- **Agent 1**: Automotive Coding Standards Analyzer
- **Agent 2**: Unit Test Generator
- **Authentication**: Amazon Cognito with JWT tokens
- **Runtime**: Amazon Bedrock AgentCore Runtime
- **Model**: Amazon Nova 2 Lite

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Multi-agent automotive analysis                                                  |
| Agent type          | Graph pattern with 2 agents                                                     |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Amazon Nova 2 Lite                                                     |
| Tutorial components | Automotive coding standards analysis, Unit test generation, AgentCore Runtime  |
| Tutorial vertical   | Automotive                                                                       |
| Example complexity  | Intermediate                                                                     |
| Inbound Auth        | Cognito                                                                          |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                    |

## Prerequisites

To execute this tutorial you will need:
* Python 3.13+ (Recommended: virtual environment at workspace level created and top level requirements installed)
* AWS credentials configured for a US based region

In [ ]:
# Optionally set your AWS_PROFILE and AWS_REGION (region has to be US based)
import os
# os.environ['AWS_PROFILE'] = 'default'
# os.environ['AWS_REGION'] = 'us-west-2'

In [ ]:
# Optionally check that required packages are installed
!pip install --force-reinstall -r ../c-code-analyzer-agent/requirements.txt --quiet

## Setting up Amazon Cognito for Authentication

Let's provision a Cognito User Pool with an App client and one test user for automotive applications.

In [ ]:
import sys
import os
import time

from utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool for automotive applications...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed")

## Preparing the Automotive C Code Analyzer Agent Graph

Let's examine our multi-agent graph system that implements custom automotive coding standards compliance checking and conditional unit test generation using the Strands GraphBuilder pattern.

### Graph Architecture

Our system uses a **conditional sequential graph** pattern:

```
┌─────────────────────┐    Condition: No HIGH    ┌─────────────────────┐
│ Automotive Analyzer │────severity violations───▶│  Unit Test Generator │
│   (Entry Point)     │                          │  (Conditional Node)  │
└─────────────────────┘                          └─────────────────────┘
```

- **Node 1**: Automotive Coding Standards Analyzer analyzes code for violations
- **Conditional Edge**: Only proceeds to Node 2 if no severe violations found
- **Node 2**: Unit Test Generator creates comprehensive tests (only if condition met)

This ensures that unit tests are only generated for code that meets automotive safety standards.

## Deploying the Agent to AgentCore Runtime

Now we'll configure and deploy our automotive C code analyzer to AgentCore Runtime with inbound authentication.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url")
client_id = cognito_config.get("client_id")

print(f"Discovery URL: {discovery_url}")
print(f"Client ID: {client_id}")

### Configure AgentCore Runtime Deployment

In [ ]:
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="../c-code-analyzer-agent/c_code_analyzer.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="../c-code-analyzer-agent/requirements.txt",
    region=region,
    agent_name="automotive_c_analyzer",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
print("Configuration response:", response)

### Launch Agent to AgentCore Runtime

In [ ]:
print("Launching automotive C code analyzer to AgentCore Runtime...")
launch_result = agentcore_runtime.launch()
print("Launch result:", launch_result)

### Check AgentCore Runtime Status

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print(f"Initial status: {status}")
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

print(f"Final status: {status}")

## Testing the Automotive C Code Analyzer

Let's test our multi-agent system with sample C code that has automotive coding standards violations.

In [ ]:
# Define a method to parse and display agent responses more beautifully
import re
from IPython.display import display, Markdown

def parse_agent_response(response_str):
    """Parse agent response and extract markdown content."""
    
    def extract_text(text):
        """Extract and unescape text content."""
        # Unescape newlines and quotes
        text = text.replace('\\\\n', '\n')
        text = text.replace("\\\\'" , "'")
        text = text.replace('\\\\"', '"')
        return text
    
    # Find text content in the response
    match = re.search(r"'text': '(.*?)'\}]}", response_str, re.DOTALL)
    if match:
        return extract_text(match.group(1))
    return "Could not parse response"

### Test Case 1: C Code with Severe Automotive Standards Violations

In [ ]:
# Load sample C code with severe automotive standards violations from file
with open('../sample-data/sample-c-code/sample_with_severe_violations.c', 'r') as f:
    c_code_with_violations = f.read()

print("Sample C code with severe violations:")
print("=" * 50)
print(c_code_with_violations[:500] + "...")
print("=" * 50)

bearer_token = reauthenticate_user(cognito_config.get("client_id"))

print("\nTesting C code with severe automotive standards violations...")
invoke_response = agentcore_runtime.invoke(
    {"c_code": c_code_with_violations}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)
response_text = parse_agent_response(invoke_response['response'])
display(Markdown(response_text))

### Test Case 2: C Code with Minor Automotive Standards Violations

In [ ]:
with open('../sample-data/sample-c-code/sample_with_minor_violations.c', 'r') as f:
    c_code_minor_violations = f.read()

print("Sample C code with minor violations:")
print("=" * 50)
print(c_code_minor_violations[:500] + "...")
print("=" * 50)

print("\nTesting C code with minor automotive standards violations...")
invoke_response = agentcore_runtime.invoke(
    {"c_code": c_code_minor_violations}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)
response_text = parse_agent_response(invoke_response['response'])
display(Markdown(response_text))

### Test Case 3: Automotive Standards-Compliant C Code

In [ ]:
with open('../sample-data/sample-c-code/sample_automotive_compliant.c', 'r') as f:
    c_code_compliant = f.read()

print("Sample automotive standards-compliant C code:")
print("=" * 50)
print(c_code_compliant[:500] + "...")
print("=" * 50)

print("\nTesting automotive standards-compliant C code...")
invoke_response = agentcore_runtime.invoke(
    {"c_code": c_code_compliant}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)
response_text = parse_agent_response(invoke_response['response'])
display(Markdown(response_text))

## Testing Without Authorization (Should Fail)

In [ ]:
# This should fail with AccessDeniedException
try:
    print("Testing without authorization (should fail)...")
    invoke_response = agentcore_runtime.invoke({"c_code": c_code_compliant})
    print("Unexpected success:", invoke_response)
except Exception as e:
    print(f"Expected error: {e}")

## Cleanup (Optional)

Clean up the AgentCore Runtime and ECR repository when done testing.

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# import boto3

# agentcore_control_client = boto3.client(
#     'bedrock-agentcore-control',
#     region_name=region
# )
# ecr_client = boto3.client(
#     'ecr',
#     region_name=region
# )

# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,
# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1].split(':')[0],
#     force=True
# )

# print("Cleanup Done.")

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# import os

# if os.path.exists('.bedrock_agentcore.yaml'):
#     os.remove('.bedrock_agentcore.yaml')
#     print("Deleted .bedrock_agentcore.yaml file")

# print("Cleanup of local files done.")

## Summary

This notebook demonstrated:

1. **Multi-Agent Graph Architecture**: Implemented using Strands GraphBuilder with conditional execution
   - Automotive Coding Standards Analyzer (Entry Node)
   - Unit Test Generator (Conditional Node)
   - Conditional edge based on violation severity

2. **Conditional Workflow**: Unit tests are only generated if no severe MISRA violations are found
   - Graph execution metrics and node tracking
   - Automatic workflow control based on analysis results

3. **Automotive Focus**: Specialized for automotive C code analysis with safety standards
   - Custom automotive coding standards compliance checking
   - Safety-critical code validation

4. **Security**: Implemented inbound authentication using Amazon Cognito
   - JWT token validation
   - Secure multi-agent execution

5. **Scalability**: Deployed on Amazon Bedrock AgentCore Runtime for production use
   - Graph-based agent orchestration
   - Performance monitoring and metrics

The system provides a comprehensive solution for automotive software development, using advanced multi-agent graph patterns to ensure code quality and safety compliance before proceeding with test generation.